# TTS Text Optimizer — Qwen3:14b (with Overlapping Chunks)

**Format translated Hindi text for optimal DesiVocal.com TTS output using qwen3:14b reasoning.**

This notebook:
1. **Installs Ollama** and downloads `qwen3:14b`
2. **Shows the model's reasoning process** in real-time as it analyzes your text
3. **Splits text into overlapping chunks** for cross-boundary context continuity
4. **Formats text** with proper speaker identification, punctuation, and DesiVocal-specific fixes
5. **Anti-hallucination guards** — prevents Chinese, repetition, and language switching
6. **Downloads** the optimized `.txt` file ready for TTS

Optimized for Hindi literary text (Sherlock Holmes, Ramayana, Mahabharata, Dracula, Alice in Wonderland, etc.) → natural, human-like TTS audio on DesiVocal.com.

## Step 1: Install and Setup Ollama
Run this cell to install Ollama and start the server in the background.

In [ ]:
# Install required packages
!pip install -q ollama requests ipywidgets

# Install and start Ollama server
import subprocess
import time
import os
import sys

print("Installing Ollama...")

# Install zstd first (required for Ollama extraction)
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1

# Download and install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("\nStarting Ollama server in background...")

# Start Ollama server in background
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(['/usr/local/bin/ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for server to start
time.sleep(5)

# Verify server is running
try:
    import ollama
    ollama.list()
    print("[OK] Ollama server is running and ready!")
except Exception as e:
    print(f"[WARN] Ollama server may not be ready yet. Error: {e}")
    print("   Please wait a few seconds and try running the next cell.")

## Step 2: Download qwen3:14b
This pulls the qwen3:14b model — strong multilingual reasoning with excellent Hindi support.

**Requires ~10GB VRAM** (T4 GPU in Colab is sufficient).

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import ollama

print("Thinking Model Selection")
print("=" * 40)

# Thinking model options
THINKING_MODELS = {
    "qwen3:14b (Strong Multilingual Thinking)": "qwen3:14b",
}

model_dropdown = widgets.Dropdown(
    options=list(THINKING_MODELS.keys()),
    value="qwen3:14b (Strong Multilingual Thinking)",
    description='Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px')
)

display(model_dropdown)
print("\nSelect a model and run the next cell to download it.")
print("NOTE: 14b models need ~10GB VRAM.")

In [ ]:
# Pull the selected thinking model
selected_model_name = THINKING_MODELS[model_dropdown.value]
print(f"Downloading model: {selected_model_name}...")
print("This may take several minutes for large models.")

try:
    current_digest = ''
    for progress in ollama.pull(selected_model_name, stream=True):
        digest = progress.get('digest', '')
        if digest != current_digest and current_digest:
             print()
        current_digest = digest

        status = progress.get('status', '')
        if 'completed' in progress and 'total' in progress:
             completed = progress['completed']
             total = progress['total']
             pct = (completed / total * 100) if total > 0 else 0
             print(f"\r   {status}: {pct:.1f}%", end='', flush=True)
        else:
             print(f"\r   {status}", end='', flush=True)

    print(f"\n\n[OK] Model '{selected_model_name}' ready to use!")
except Exception as e:
    print(f"\n[ERROR] Error pulling model: {e}")

## Step 3: TTS Optimizer Engine (with Overlapping Chunks & Anti-Hallucination)
Defines the optimizer class with:
- **Overlapping chunks** — each chunk includes context from the previous one for continuity
- **Comprehensive Hindi TTS prompt** — tailored for DesiVocal.com
- **Anti-hallucination guards** — prevents Chinese, repetition, and language switching
- **Streaming thinking display** — shows qwen3:14b reasoning in real-time

In [ ]:
import requests
import json
import sys
import re
import time
from IPython.display import display, HTML, clear_output

# ══════════════════════════════════════════════════════════════
# KNOWN THINKING MODEL PATTERNS
# ══════════════════════════════════════════════════════════════
# Different thinking models use different tag patterns for their reasoning:
#   deepseek-r1  : <think> ... </think>
#   qwen3        : <think> ... </think>
#   magistral    : [Thinking] ... [/Thinking] or just outputs reasoning first
#   gpt-oss      : <|begin_of_thought|> ... <|end_of_thought|>
# We handle all of these patterns.

THINK_START_PATTERNS = ['<think>', '<|begin_of_thought|>', '[Thinking]']
THINK_END_PATTERNS = ['</think>', '<|end_of_thought|>', '[/Thinking]']

# Context overlap markers
OVERLAP_START_TAG = '[CONTEXT_FROM_PREVIOUS]'
OVERLAP_END_TAG = '[END_CONTEXT]'


class TTSThinkingOptimizer:
    """Optimizes Hindi text for DesiVocal.com TTS using qwen3:14b with overlapping chunks."""

    def __init__(self, model_name="qwen3:14b", chunk_size=1500, overlap_size=200, timeout=6000):
        self.ollama_url = "http://localhost:11434/api/generate"
        self.model = model_name
        self.chunk_size = chunk_size
        self.overlap_size = overlap_size
        self.timeout = timeout
        print(f"Initialized TTS Thinking Optimizer")
        print(f"   Model: {self.model}")
        print(f"   Chunk size: {self.chunk_size} chars")
        print(f"   Overlap size: {self.overlap_size} chars")
        print(f"   Timeout: {self.timeout}s per chunk")

    def chunk_text(self, text: str) -> list:
        """
        Split text into chunks at paragraph/sentence boundaries.
        Returns list of dicts: {'text': str, 'overlap_prefix': str}
        overlap_prefix is the tail of the previous chunk (for context).
        """
        if len(text) <= self.chunk_size:
            return [{'text': text, 'overlap_prefix': ''}]

        # First, build raw chunks at paragraph/sentence boundaries
        raw_chunks = []
        current_chunk = ""

        # Split by paragraphs (double newline)
        paragraphs = text.split('\n\n')

        for para in paragraphs:
            if len(current_chunk) + len(para) + 2 > self.chunk_size and current_chunk:
                raw_chunks.append(current_chunk.strip())
                current_chunk = para
            else:
                current_chunk += ("\n\n" if current_chunk else "") + para

        if current_chunk.strip():
            raw_chunks.append(current_chunk.strip())

        # If any chunk is still too large, split by sentences
        split_chunks = []
        for chunk in raw_chunks:
            if len(chunk) <= self.chunk_size:
                split_chunks.append(chunk)
            else:
                sentences = re.split(r'([\u0964\u0965.!?]\s+)', chunk)
                sub_chunk = ""
                for i in range(0, len(sentences), 2):
                    sentence = sentences[i]
                    separator = sentences[i+1] if i+1 < len(sentences) else ""
                    if len(sub_chunk) + len(sentence) + len(separator) > self.chunk_size and sub_chunk:
                        split_chunks.append(sub_chunk.strip())
                        sub_chunk = sentence + separator
                    else:
                        sub_chunk += sentence + separator
                if sub_chunk.strip():
                    split_chunks.append(sub_chunk.strip())

        # Fallback: if still no chunks, force-split
        if not split_chunks:
            split_chunks = [text[i:i+self.chunk_size] for i in range(0, len(text), self.chunk_size)]

        # Now build overlapping chunks
        overlapping_chunks = []
        for idx, chunk in enumerate(split_chunks):
            if idx == 0:
                overlap_prefix = ''
            else:
                prev_chunk = split_chunks[idx - 1]
                if self.overlap_size > 0 and len(prev_chunk) > 0:
                    # Take the tail of previous chunk, but try to start at a sentence boundary
                    tail = prev_chunk[-self.overlap_size:]
                    # Try to find a sentence start within the tail
                    sentence_break = re.search(r'[\u0964\u0965.!?]\s+', tail)
                    if sentence_break:
                        tail = tail[sentence_break.end():]
                    overlap_prefix = tail.strip()
                else:
                    overlap_prefix = ''

            overlapping_chunks.append({
                'text': chunk,
                'overlap_prefix': overlap_prefix
            })

        print(f"\nText split into {len(overlapping_chunks)} chunks (overlap: {self.overlap_size} chars)")
        for idx, chunk_data in enumerate(overlapping_chunks, 1):
            overlap_info = f" + {len(chunk_data['overlap_prefix'])} chars overlap" if chunk_data['overlap_prefix'] else ""
            print(f"   Chunk {idx}: {len(chunk_data['text'])} chars{overlap_info}")

        return overlapping_chunks

    def get_optimization_prompt(self, text: str, overlap_prefix: str = '', chunk_num: int = 1, total_chunks: int = 1) -> str:
        """Build the TTS optimization prompt for qwen3:14b."""

        # Build overlap context section if applicable
        overlap_section = ""
        if overlap_prefix:
            overlap_section = f"""
OVERLAP CONTEXT (from previous chunk — DO NOT re-output this, use for context only):
{OVERLAP_START_TAG}
{overlap_prefix}
{OVERLAP_END_TAG}

The above overlap text has ALREADY been formatted in the previous chunk's output.
Use it ONLY to understand who was speaking, what the context was, and maintain continuity.
Do NOT include this overlap text in your output. Start your output from the NEW TEXT below.

"""

        chunk_info = ""
        if total_chunks > 1:
            chunk_info = f"\nYou are processing chunk {chunk_num} of {total_chunks}. Maintain consistency with previous chunks.\n"

        prompt = f"""You are an expert Hindi text formatter for DesiVocal.com TTS system.

CRITICAL RULES FOR YOUR OUTPUT:
- Output ONLY the formatted Hindi text. Nothing else.
- NO Chinese text. NO English explanations. NO meta-commentary.
- NO repetition of sentences. If you notice yourself repeating, STOP.
- NEVER write "Here is the formatted text" or similar prefixes.
- NEVER include your reasoning/thinking steps in the output.
- Keep EVERY word from the input (except attribution words that become speaker tags).
{chunk_info}
YOUR TASK:
Format Hindi text for DesiVocal.com — a SINGLE-VOICE TTS system, NO SSML support.
One voice reads everything. Listeners cannot tell speakers apart unless you mark them.

SPEAKER IDENTIFICATION (use your reasoning internally):
- "Holmes ne kaha" → Speaker is Holmes
- "raj ne poocha" → Speaker is raj
- "maine kaha" → Speaker is narrator
- "usne kaha" → Look back 1-3 sentences to identify who
- Speakers typically alternate in dialogue
- Named characters (Holmes, Watson, Ram, Sita) take priority
- Roles/titles (raja, doctor, maharaj) as secondary
- Last resort only: vakta1, vakta2

DIALOGUE PUNCTUATION (each character gets CONSISTENT unique marks):
- Main protagonist → 'single quotes'
- Secondary/narrator → "double quotes"
- Authority/client → *asterisks*
- Others → <<guillemets>>

FORMAT: SpeakerName: [mark]dialogue[mark]
EXAMPLE:
Holmes: 'yah zaroori hai.'
Watson: "samajh gaya."
raja: *madad karo.*

REMOVE ATTRIBUTION: "Holmes ne kaha" → "Holmes:"
SEPARATION: Each speaker on new line with blank line before.

TECHNICAL FORMATTING RULES:
1. ROMAN NUMERALS → NUMBERS: I→1, II→2, III→3, IV→4, V→5 etc.
2. NUMBERS: Remove commas (50,000 → 50000)
3. DATES: Use month names (15/03/2024 → 15 March 2024)
4. YEARS: Add "san" prefix (1988 → san1988)
5. TIME: Write in words (3:30 → saadhe teen, 10:00 → ten baje)
6. THE "10" BUG — CRITICAL: DesiVocal cannot speak "10" or "das" properly.
   ALWAYS replace with "ten" (10 books → ten kitaabein, Chapter 10 → Chapter ten)
7. RANGES: Use "se" (5-8 → 5se8, 10-15 → tense15)
8. PERCENTAGES: 50% → 50 percent
9. ABBREVIATIONS: Dr. → Doctor, Rs. → rupaye, km → kilometer
10. ACRONYMS: Remove periods (U.S.A. → USA)
11. EMAILS/URLs: @ → at the rate, . → dot (hr@company.com → hr at the rate company dot com)
12. HYPHENS: Remove from compounds (cross-check → cross check)
13. SYMBOLS: °F → degree Fahrenheit, × → guna

PACING PUNCTUATION (non-dialogue narration):
, = short pause
| = medium pause (context shift)
. = long pause (sentence end)
,, = extended pause
... = suspense
!! = excitement
?? = confusion

Example: "usne khana khaya. | fir so gaya. | subah utha."

EXAMPLE 1 (Sherlock Holmes):
INPUT: adhyay I
yah 15 march, 1988 ki baat hai. Holmes ne kaha, "main 10 baje aaunga." Watson ne poocha, "kyun?" "kyunki yah zaroori hai," Holmes ne kaha.

OUTPUT:
adhyay 1.

yah 15 march, san1988 ki baat hai.

Holmes: 'main ten baje aaunga.'

Watson: "kyun?"

Holmes: 'kyunki yah zaroori hai.'

EXAMPLE 2 (Pronoun resolution):
INPUT: Holmes kamre mein khada tha. Watson darwaze par aaya. usne poocha, "kya hua?" "kuch nahi," usne kaha.
OUTPUT:
Holmes kamre mein khada tha. | Watson darwaze par aaya.

Watson: "kya hua?"

Holmes: 'kuch nahi.'

ANTI-HALLUCINATION CHECKS (verify before outputting):
- NO Chinese characters anywhere
- NO repeated sentences
- ALL text is Hindi (Devanagari or Romanized Hindi)
- NO reasoning steps shown
- NO English explanations
- All Roman numerals converted
- All "10"/"das" → "ten"
- All years have "san" prefix
- Each character has consistent punctuation marks
{overlap_section}
NEW TEXT TO FORMAT:
{text}"""
        return prompt

    def optimize_chunk_streaming(self, chunk_data: dict, chunk_num: int = 1, total_chunks: int = 1, retry_count: int = 3) -> str:
        """
        Optimize a single chunk with streaming output showing thinking process.
        chunk_data: {'text': str, 'overlap_prefix': str}
        """
        prompt = self.get_optimization_prompt(
            chunk_data['text'],
            chunk_data.get('overlap_prefix', ''),
            chunk_num,
            total_chunks
        )

        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": True,
            "options": {
                "temperature": 0.15,
                "top_p": 0.85,
                "top_k": 30,
                "repeat_penalty": 1.3,
                "num_predict": -1
            }
        }

        for attempt in range(retry_count):
            try:
                response = requests.post(
                    self.ollama_url,
                    json=payload,
                    stream=True,
                    timeout=self.timeout
                )
                response.raise_for_status()

                full_response = ""
                thinking_content = ""
                output_content = ""
                in_thinking = False
                thinking_started = False
                thinking_ended = False

                overlap_info = f" | overlap: {len(chunk_data.get('overlap_prefix', ''))} chars" if chunk_data.get('overlap_prefix') else ""
                print(f"\n{'='*60}")
                print(f"  CHUNK {chunk_num}/{total_chunks} ({len(chunk_data['text'])} chars{overlap_info})")
                print(f"{'='*60}")

                for line in response.iter_lines():
                    if not line:
                        continue
                    try:
                        data = json.loads(line)
                        token = data.get("response", "")
                        full_response += token

                        # Detect thinking start
                        for pattern in THINK_START_PATTERNS:
                            if pattern in full_response and not thinking_started:
                                thinking_started = True
                                in_thinking = True
                                print(f"\n--- MODEL REASONING ---")
                                # Remove the tag from display
                                remaining = full_response.split(pattern, 1)[-1]
                                if remaining:
                                    thinking_content += remaining
                                    sys.stdout.write(remaining)
                                    sys.stdout.flush()
                                full_response = ""  # Reset to avoid re-matching
                                break

                        # Detect thinking end
                        if in_thinking:
                            for pattern in THINK_END_PATTERNS:
                                if pattern in token:
                                    in_thinking = False
                                    thinking_ended = True
                                    # Get any content before the end tag
                                    before_tag = token.split(pattern)[0]
                                    if before_tag:
                                        thinking_content += before_tag
                                        sys.stdout.write(before_tag)
                                        sys.stdout.flush()
                                    print(f"\n--- END REASONING ---\n")
                                    print(f"--- FORMATTED OUTPUT ---")
                                    # Get content after end tag
                                    after_tag = token.split(pattern, 1)[-1]
                                    if after_tag.strip():
                                        output_content += after_tag
                                        sys.stdout.write(after_tag)
                                        sys.stdout.flush()
                                    break
                            else:
                                if in_thinking:
                                    thinking_content += token
                                    sys.stdout.write(token)
                                    sys.stdout.flush()
                        elif thinking_ended:
                            # We are past thinking, collecting output
                            output_content += token
                            sys.stdout.write(token)
                            sys.stdout.flush()
                        elif not thinking_started:
                            # Model might not use thinking tags, treat as direct output
                            output_content += token
                            sys.stdout.write(token)
                            sys.stdout.flush()

                        if data.get("done", False):
                            break
                    except json.JSONDecodeError:
                        continue

                print(f"\n{'='*60}")

                # If we never entered thinking mode, the full response is the output
                if not thinking_started:
                    output_content = full_response

                # Clean the output
                cleaned = self._clean_output(output_content)

                # Validate output quality
                warnings = self._validate_output(cleaned, chunk_data['text'])
                if warnings:
                    for w in warnings:
                        print(f"[WARN] {w}")

                print(f"[OK] Chunk {chunk_num}/{total_chunks} complete! ({len(cleaned)} chars output)")

                return cleaned

            except requests.exceptions.Timeout:
                if attempt < retry_count - 1:
                    wait_time = (attempt + 1) * 10
                    print(f"\n[WARN] Timeout on attempt {attempt + 1}/{retry_count}. Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"\n[ERROR] Failed after {retry_count} attempts due to timeout")
                    raise
            except Exception as e:
                if attempt < retry_count - 1:
                    wait_time = (attempt + 1) * 5
                    print(f"\n[WARN] Error on attempt {attempt + 1}/{retry_count}: {e}")
                    print(f"   Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"\n[ERROR] Failed after {retry_count} attempts: {e}")
                    raise

        return None

    def optimize(self, text: str) -> str:
        """Optimize text with overlapping chunks, streaming thinking display."""
        chunk_data_list = self.chunk_text(text)

        if len(chunk_data_list) == 1:
            print(f"\nProcessing single chunk ({len(text)} chars)...")
            return self.optimize_chunk_streaming(chunk_data_list[0], 1, 1)

        print(f"\nProcessing {len(chunk_data_list)} overlapping chunks with visible reasoning...")
        optimized_chunks = []

        for idx, chunk_data in enumerate(chunk_data_list, 1):
            try:
                optimized = self.optimize_chunk_streaming(chunk_data, idx, len(chunk_data_list))
                if optimized:
                    optimized_chunks.append(optimized)
                else:
                    print(f"[WARN] Chunk {idx}/{len(chunk_data_list)} failed - using original")
                    optimized_chunks.append(chunk_data['text'])
            except Exception as e:
                print(f"[ERROR] Error processing chunk {idx}: {e}")
                print("   Using original chunk text")
                optimized_chunks.append(chunk_data['text'])

        final_text = "\n\n".join(optimized_chunks)
        print(f"\n[OK] All chunks processed! Total output: {len(final_text)} characters")
        return final_text

    def _clean_output(self, text: str) -> str:
        """Clean the model output — remove formatting artifacts, overlap tags, and hallucinations."""
        # Remove markdown artifacts
        text = text.replace("```", "").replace("**", "")

        # Remove any remaining thinking tags
        for pattern in THINK_START_PATTERNS + THINK_END_PATTERNS:
            text = text.replace(pattern, "")

        # Remove overlap context tags and their content (if model leaked them)
        overlap_pattern = re.compile(
            re.escape(OVERLAP_START_TAG) + r'.*?' + re.escape(OVERLAP_END_TAG),
            re.DOTALL
        )
        text = overlap_pattern.sub('', text)
        # Also remove standalone tags
        text = text.replace(OVERLAP_START_TAG, '').replace(OVERLAP_END_TAG, '')

        # Remove Chinese characters (hallucination guard)
        chinese_chars = re.findall(r'[\u4e00-\u9fff\u3400-\u4dbf]+', text)
        if chinese_chars:
            print(f"[WARN] Removed {len(chinese_chars)} Chinese text segments (hallucination detected)")
            text = re.sub(r'[\u4e00-\u9fff\u3400-\u4dbf]+', '', text)

        # Remove lines that look like metadata/headers/explanations
        lines = []
        for line in text.split('\n'):
            stripped = line.strip()
            if not stripped:
                lines.append('')
                continue
            # Skip meta lines
            if stripped.startswith('#') or stripped.startswith('OUTPUT'):
                continue
            # Skip lines that are just dashes or equals
            if re.match(r'^[-=]{3,}$', stripped):
                continue
            # Skip English explanation prefixes
            skip_prefixes = [
                'Here is', 'Below is', 'The formatted', 'I will', 'Let me',
                'First,', 'Note:', 'Formatted text:', 'Output:',
                'NEW TEXT', 'CONTEXT_FROM', 'END_CONTEXT'
            ]
            if any(stripped.startswith(prefix) for prefix in skip_prefixes):
                continue
            lines.append(line.rstrip())

        return '\n'.join(lines).strip()

    def _validate_output(self, output: str, original: str) -> list:
        """Validate output quality and return warnings."""
        warnings = []

        # Check for Chinese characters
        if re.search(r'[\u4e00-\u9fff]', output):
            warnings.append("Chinese characters detected in output — possible hallucination")

        # Check for excessive repetition (same line 3+ times)
        lines = [l.strip() for l in output.split('\n') if l.strip()]
        from collections import Counter
        line_counts = Counter(lines)
        for line, count in line_counts.items():
            if count >= 3 and len(line) > 10:  # Ignore short common lines
                warnings.append(f"Repetition detected: '{line[:50]}...' appears {count} times")

        # Check output length ratio (should be roughly similar to input)
        if len(original) > 100:
            ratio = len(output) / len(original)
            if ratio > 3.0:
                warnings.append(f"Output is {ratio:.1f}x longer than input — possible hallucination")
            elif ratio < 0.2:
                warnings.append(f"Output is only {ratio:.1%} of input length — possible truncation")

        return warnings


print("[OK] TTSThinkingOptimizer loaded with overlapping chunks & anti-hallucination!")

## Step 4: Upload Text and Configure
Upload your `.txt` file and configure chunk size and overlap.

**Overlap** = how many characters from the end of the previous chunk are passed as context to the next chunk.
This helps the model maintain speaker continuity and context across chunk boundaries.

In [ ]:
from google.colab import files
import ipywidgets as widgets
from IPython.display import display

print("Upload your text file (.txt):")
uploaded = files.upload()

if uploaded:
    uploaded_filename = list(uploaded.keys())[0]
    file_size = len(uploaded[uploaded_filename])
    print(f"[OK] Uploaded: {uploaded_filename} ({file_size:,} bytes)")
else:
    print("[WARN] No file uploaded yet.")

# Chunk size selector
chunk_size_input = widgets.IntSlider(
    value=1500,
    min=500,
    max=3000,
    step=100,
    description='Chunk Size:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Overlap size selector
overlap_input = widgets.IntSlider(
    value=200,
    min=0,
    max=500,
    step=50,
    description='Overlap Size:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

print("\nConfiguration:")
display(chunk_size_input)
display(overlap_input)
print("\nChunk size guide: smaller = more API calls but less hallucination risk")
print("   Recommended: 1200-1500 for qwen3:14b on Hindi text")
print("\nOverlap guide: more overlap = better cross-chunk context")
print("   Recommended: 150-250 chars (set to 0 to disable)")

## Step 5: Run Optimization (with Visible Thinking)
The model will show its reasoning process as it analyzes and formats each chunk.
You will see:
- `--- MODEL REASONING ---` : qwen3:14b thinking through speaker identification, context
- `--- FORMATTED OUTPUT ---` : The formatted Hindi text for DesiVocal TTS
- `[WARN]` : Any hallucination warnings (Chinese text, repetition, etc.)

In [ ]:
# Run Optimization with Thinking Display
if not uploaded:
    print("[WARN] Please upload a file in the previous step first!")
else:
    try:
        text_content = uploaded[uploaded_filename].decode("utf-8")
        print(f"Read {len(text_content):,} characters from file.")

        # Initialize optimizer with selected model
        try:
            model_to_use = selected_model_name
        except NameError:
            model_to_use = "qwen3:14b"  # Fallback
            print("[WARN] Using default model: qwen3:14b")

        optimizer = TTSThinkingOptimizer(
            model_name=model_to_use,
            chunk_size=chunk_size_input.value,
            overlap_size=overlap_input.value,
            timeout=6000  # 100 minutes per chunk for thinking models
        )

        print(f"\nStarting optimization with {model_to_use}...")
        print(f"Parameters: temp=0.15, top_p=0.85, top_k=30, repeat_penalty=1.3")
        print("The model's reasoning process will be displayed below.")
        print("=" * 60)

        start_time = time.time()
        optimized_text = optimizer.optimize(text_content)
        end_time = time.time()

        processing_time = end_time - start_time

        if optimized_text:
            print("\n" + "=" * 60)
            print("OPTIMIZATION COMPLETE!")
            print("=" * 60)
            print(f"Processing time: {processing_time:.1f} seconds ({processing_time/60:.1f} minutes)")
            print(f"Input length: {len(text_content):,} chars")
            print(f"Output length: {len(optimized_text):,} chars")
            print(f"Size change: {((len(optimized_text) - len(text_content)) / len(text_content) * 100):+.1f}%")

            print("\nPreview (First 800 characters):")
            print("=" * 60)
            print(optimized_text[:800])
            if len(optimized_text) > 800:
                print("\n... (truncated)")
            print("=" * 60)

            # Save to file
            output_filename = f"tts_optimized_{uploaded_filename}"
            with open(output_filename, 'w', encoding='utf-8') as f:
                f.write(optimized_text)

            print(f"\nSaved to: {output_filename}")
            print("Downloading file...")

            # Trigger download
            files.download(output_filename)
            print("\n[OK] Done! Check your downloads folder.")

        else:
            print("\n[ERROR] Optimization failed. Please check the errors above.")

    except Exception as e:
        print(f"\n[ERROR] Error reading or processing file: {e}")
        import traceback
        print("\nFull error details:")
        print(traceback.format_exc())

## Troubleshooting

**If you get timeouts:**
1. Reduce chunk size to 1000-1200 characters
2. Set overlap to 100-150 (smaller overlap = faster)
3. Check Ollama server: `!ollama ps`
4. Restart Ollama: Go back to Step 1 and re-run

**If you see Chinese text in output:**
- The anti-hallucination system will auto-remove it
- If it persists, reduce chunk size further (smaller chunks = less hallucination)
- The `repeat_penalty=1.3` and `temperature=0.15` settings are specifically tuned to prevent this

**If output repeats:**
- The validator will warn you about repetition
- Reduce chunk size to 1000 chars
- `repeat_penalty=1.3` directly combats this

**Overlap explained:**
- When text is split into chunks, each chunk (except the first) receives the last ~200 characters from the previous chunk as context
- This helps the model maintain speaker identity and narrative continuity across chunk boundaries
- The context is marked so the model doesn't re-output it (no duplicates)
- Set overlap to 0 to disable (faster but may lose context)

**Thinking models are slower** than regular models because they reason through the text first. This is expected and produces better results for:
- Complex dialogue with many speakers
- Pronoun resolution in long passages
- Genre-appropriate formatting

**For very large files (100k+ chars):**
- Use chunk size of 1000 characters with 150 overlap
- Processing will take longer but will be more reliable

**Parameter reference:**
| Parameter | Value | Purpose |
|---|---|---|
| temperature | 0.15 | Very low = deterministic, no hallucination |
| top_p | 0.85 | Tight nucleus sampling |
| top_k | 30 | Limits token candidates |
| repeat_penalty | 1.3 | Penalizes repeated tokens/phrases |